# Figure 5

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(tidyverse)
  library(colocboost)
  library(BEDMatrix)
})
gene_id = 'ENSG00000159640'
gene_name = 'ACE'

## Reference annotation

In [ ]:
anno_ref <- fread( 'Annotation_with_gp_coordinates_clean.txt', sep = '\t')

## Data caches for the panels b, c

### b colocBoost locus object and gene track (`data/Figure_5a.rds`)

In [ ]:
cb_res_paths = list.files('data_output/colocalization/rds_files/cis', 'AD_Bellenguez.rds', full.names = T)
target_paths <- cb_res_paths[str_detect(cb_res_paths, gene_id)] %>% .[str_detect(., 'unadj')]
target_paths

cb_res <- readRDS(target_paths[1])
cb_res[[1]]$cos_summary

start = 3900
end = 4400
cb_res[[1]]$data_info$outcome_info$outcome_names

# replace ID in cb plot
id <- cb_res[[1]]$data_info$outcome_info$outcome_names %>% gsub('DLPFC_Bennett_', '',.) %>% gsub('DLPFC_Klein_gpQTL_unadjusted_', '',.)

id_dt <- data.table(original_order = seq_along(id), id = id)
id_dt[, gpID := str_extract(id, "^gp_\\d+")]
gp_map <- anno_ref[, .(gpID, peptide_ID)]
id_dt <- merge(id_dt, gp_map, by = "gpID", all.x = TRUE)
id_dt[, new_id := fifelse(!is.na(peptide_ID), peptide_ID, id)]
setorder(id_dt, original_order)
new_id_vector <- id_dt$new_id
new_id_vector <- gsub(gene_id, gene_name, new_id_vector) %>% gsub('pQTL_', 'Protein ', .)

region_chr = cb_res[[1]]$data_info$variables[start] %>% str_split(., ':', simplify = T) %>% .[,1]
region_start = cb_res[[1]]$data_info$variables[start] %>% str_split(., ':', simplify = T) %>% .[,2] %>% as.numeric
region_end = cb_res[[1]]$data_info$variables[end] %>% str_split(., ':', simplify = T) %>% .[,2] %>% as.numeric
region = paste0(region_chr, ':', region_start, '-', region_end)
region

In [ ]:
suppressPackageStartupMessages(library(ggbio))
# ggbio and ggplot2 export identically named functions
suppressMessages({
conflicts_prefer(ggplot2::autoplot)
conflicts_prefer(ggplot2::geom_bar)
conflicts_prefer(ggplot2::geom_rect)
conflicts_prefer(ggplot2::geom_segment)
conflicts_prefer(ggplot2::ggsave)
conflicts_prefer(ggplot2::stat_bin)
conflicts_prefer(ggplot2::stat_identity)
conflicts_prefer(ggplot2::xlim)
})
suppressPackageStartupMessages(library(EnsDb.Hsapiens.v86))
suppressPackageStartupMessages(library(ensembldb))
suppressPackageStartupMessages(library(GenomicRanges))
suppressPackageStartupMessages(library(ggplot2))
region <- GRanges(region_chr %>% gsub('chr','',.), IRanges(region_start, region_end))

tx <- transcripts(EnsDb.Hsapiens.v86,
                  filter = GRangesFilter(region),
                  columns = c("tx_id","gene_id","gene_name","tx_strand"))

tx <- tx[mcols(tx)$gene_name == gene_name] 
tx <- tx[which.max(width(tx))]
strand_dir <- as.character(strand(tx))
ex_tx <- exonsBy(EnsDb.Hsapiens.v86, by = "tx",
                 filter = TxIdFilter(tx$tx_id))[[1]]
ex_df <- as.data.frame(ex_tx)

intron_df <- data.frame(
  start = head(end(ex_tx), -1),
  end   = tail(start(ex_tx), -1),
  y     = 0
)

dir.create('data', showWarnings = FALSE)
saveRDS(list(cb_res = cb_res[[1]], new_id_vector = new_id_vector, intron_df = intron_df, ex_df = ex_df) , 'data/Figure_5a.rds')

coloc_names <- new_id_vector %>%
  gsub('^Protein ACE$', 'pQTL', .) %>%
  gsub('^ACE ', '', .) %>%
  gsub('^AD_Bellenguez$', 'GWAS (AD)', .)

s5a <- list(cb_res = cb_res[[1]], coloc_names = coloc_names, start = start, end = end)
saveRDS(s5a, '../../3_supplementary_figures/Supplementary_Figure5/data/Figure_S5a.rds')

### c genotype boxplot data (`data/Figure_5c.rds`)

In [ ]:
gp_pheno <- fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/phenotype_preprocessing/unadj_glyco.sorted.bed.gz')
p_pheno <- fread('ftp_fgc_xqtl/pQTL/ROSMAP/DLPFC/analysis_ready/phenotype_preprocessing/protein.sorted.bed.gz')

geno_chr17 <- suppressMessages(BEDMatrix('FunGen_xQTL/ROSMAP/Genotype/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.17.bed'))

bim_chr17 <- fread('FunGen_xQTL/ROSMAP/Genotype/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.17.bim')
fam_chr17 <- fread('FunGen_xQTL/ROSMAP/Genotype/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.17.fam')

colnames(geno_chr17) <- bim_chr17$V2 %>% gsub("_", ":", .)
rownames(geno_chr17) <- fam_chr17$V2

# The bed files have 4 metadata columns
gpID1 = 'gp_2626'
pheno_df_gp1 <- gp_pheno %>% dplyr::filter(ID == gpID1) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = gpID1)
colnames(pheno_df_gp1)[1] <- 'pheno_value'

gpID2 = 'gp_2629'
pheno_df_gp2 <- gp_pheno %>% dplyr::filter(ID == gpID2) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = gpID2)
colnames(pheno_df_gp2)[1] <- 'pheno_value'

pID = gene_id
pheno_df_p <- p_pheno %>% dplyr::filter(ID == gene_id) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = 'pQTL')
colnames(pheno_df_p)[1] <- 'pheno_value'

pheno_df <- rbind(pheno_df_gp1, pheno_df_gp2, pheno_df_p)

top_var <- 'chr17:63470201:G:A'
var_ref <- str_split(top_var, ":", simplify = T) %>% .[,3]
var_alt <- str_split(top_var, ":", simplify = T) %>% .[,4]
geno0 <- paste(var_ref, var_ref, sep = '/')
geno1 <- paste(var_ref, var_alt, sep = '/')
geno2 <- paste(var_alt, var_alt, sep = '/')


geno_colors <- c(
  setNames("#1f78b4", geno0),
  setNames("#ff7f00", geno1),
  setNames("#33a02c", geno2)
)


geno_df <- geno_chr17[, top_var, drop = F] %>% as.data.frame %>% mutate(sample = rownames(.))
colnames(geno_df)[1] <- 'genotype'

In [ ]:
plot_df <- merge(pheno_df, geno_df)
# merge only guarantees phenotype + genotype
glyco_final_samples <- colnames(fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/covariate_preprocessing/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz'))[-1]
plot_df <- plot_df %>% filter(!grepl('^gp_', pheno) | sample %in% glyco_final_samples)
plot_df <- plot_df %>% mutate(genotype = ifelse(genotype == 0, geno0,
                                            ifelse(genotype == 1, geno1,
                                              ifelse(genotype == 2, geno2, genotype))))
plot_df$genotype <- factor(plot_df$genotype, levels = c(geno0, geno1, geno2))

plot_df <- plot_df %>% merge(., (anno_ref %>% dplyr::select(gpID, peptide_ID) %>% distinct), by.x = 'pheno', by.y = 'gpID', all.x = T) 
plot_df <- plot_df %>% mutate(peptide_ID = ifelse(is.na(peptide_ID), pheno,gsub(' ', '\n', peptide_ID)))
plot_df$peptide_ID <- factor(plot_df$peptide_ID, levels = c('pQTL', 'ACE\nHexNAc(6)Hex(4)Fuc(1)NeuAc(1)', 'ACE\nHexNAc(2)Hex(5)', 'ACE\nHexNAc(2)Hex(7)'))
GCOV <- 'ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/covariate_preprocessing/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz'
PCOV <- 'data_input/protein/covariate/protein.sorted.covariates_protein.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz'
read_cov <- function(f) { x <- fread(f); m <- t(as.matrix(x[, -1])); colnames(m) <- x[[1]]; m[, apply(m, 2, sd) > 0, drop = FALSE] }   # samples x covariates, zero-variance rows dropped
cov_gp <- read_cov(GCOV); cov_p <- read_cov(PCOV); stopifnot(!anyNA(cov_gp), !anyNA(cov_p)); cat('covariates glyco:', ncol(cov_gp), ' pQTL:', ncol(cov_p), '\n')
adjust_geno <- function(df, cov) {
  df <- df[df$sample %in% rownames(cov) & !is.na(df$genotype), ]; df$genotype <- droplevels(df$genotype)
  X <- cbind(1, model.matrix(~ 0 + genotype, df)[, -1, drop = FALSE], scale(cov[df$sample, , drop = FALSE]))
  fit <- lm.fit(X, df$pheno_value); b <- fit$coefficients; ok <- !is.na(b); g <- 2:nlevels(df$genotype)
  df$pheno_raw <- df$pheno_value
  df$pheno_value <- as.vector(fit$residuals + X[, g, drop = FALSE] %*% b[g]); df$pheno_value <- df$pheno_value - mean(df$pheno_value) + mean(df$pheno_raw)
  df$n_cov_used <- sum(ok) - nlevels(df$genotype); df
}
adjust_all <- function(d) do.call(rbind, lapply(split(d, d$pheno), function(x) adjust_geno(x, if (x$pheno[1] == 'pQTL') cov_p else cov_gp)))
plot_df <- adjust_all(plot_df)
stopifnot(n_distinct(plot_df$sample[grepl('^gp_', plot_df$pheno)]) == 316L, all(is.finite(plot_df$pheno_value)))

saveRDS(list(plot_df = plot_df, geno_colors = geno_colors, top_var = top_var) , 'data/Figure_5c.rds')

s5b_gp_ids <- paste0('gp_', 2625:2632)
s5b_pheno_df_gp <- purrr::map_dfr(s5b_gp_ids, function(gpid) {
  gp_pheno %>% dplyr::filter(ID == gpid) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>%
    mutate(sample = rownames(.), pheno = gpid) %>% dplyr::rename(pheno_value = 1)
})
s5b_pheno_df_p <- p_pheno %>% dplyr::filter(ID == gene_id) %>% dplyr::select(-c(1:4)) %>% t() %>%
  as.data.frame %>% mutate(sample = rownames(.), pheno = 'pQTL') %>% dplyr::rename(pheno_value = 1)
s5b_pheno_df <- rbind(s5b_pheno_df_gp, s5b_pheno_df_p)

s5b_plot_df <- merge(s5b_pheno_df, geno_df)
s5b_plot_df <- s5b_plot_df %>% filter(!grepl('^gp_', pheno) | sample %in% glyco_final_samples)
s5b_plot_df <- s5b_plot_df %>% mutate(genotype = ifelse(genotype == 0, geno0,
                                            ifelse(genotype == 1, geno1,
                                              ifelse(genotype == 2, geno2, genotype))))
s5b_plot_df$genotype <- factor(s5b_plot_df$genotype, levels = c(geno0, geno1, geno2))
s5b_plot_df <- adjust_all(s5b_plot_df)   # same covariate adjustment as panel c

# gp_2625/gp_2627/gp_2631 share HexNAc(2)Hex(5), so the gpID is part of the label
s5b_labels <- anno_ref %>% dplyr::filter(gpID %in% s5b_gp_ids) %>%
  mutate(peptide_ID = paste0(GlycanComposition, '\n(', gpID, ')')) %>%
  dplyr::select(gpID, peptide_ID)
s5b_plot_df <- s5b_plot_df %>% merge(., s5b_labels, by.x = 'pheno', by.y = 'gpID', all.x = T)
s5b_plot_df <- s5b_plot_df %>% mutate(peptide_ID = ifelse(is.na(peptide_ID), pheno, peptide_ID))

stopifnot(
  n_distinct(s5b_plot_df$pheno) == 9L,
  s5b_plot_df %>% filter(grepl('^gp_', pheno)) %>% pull(sample) %>% n_distinct() == 316L,
  all(is.finite(s5b_plot_df$pheno_value))
)
dim(s5b_plot_df)

s5b <- list(plot_df = s5b_plot_df, geno_colors = geno_colors, top_var = top_var)
saveRDS(s5b, '../../3_supplementary_figures/Supplementary_Figure5/data/Figure_S5b.rds')

## Table 2: effect size with 95% CI and P value at the lead variant

In [ ]:
# Lead variant is read from the robust object, not hardcoded, so it cannot drift
cb_res_robust5 <- get_robust_colocalization(cb_res[[1]], cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 0.0001)
cs5 <- cb_res_robust5$cos_summary
lead_var5 <- cs5$top_variable[cs5$focal_outcome == 'AD_Bellenguez'][1]
var_idx5  <- match(lead_var5, cb_res[[1]]$data_info$variables)
gp_ids5      <- c('gp_2625','gp_2626','gp_2627','gp_2628','gp_2629','gp_2630','gp_2631','gp_2632')
gp_outcomes5 <- paste0('DLPFC_Klein_gpQTL_unadjusted_', gp_ids5)

cb_tab2 <- data.table(
  molecular_trait_id = gp_ids5,
  coef_colocboost = sapply(gp_outcomes5, function(nm) cb_res[[1]]$data_info$coef[[nm]][var_idx5]),
  z_colocboost    = sapply(gp_outcomes5, function(nm) cb_res[[1]]$data_info$z[[nm]][var_idx5]))
cb_tab2[, p_colocboost := 2 * pnorm(-abs(z_colocboost))]
cb_n <- setNames(as.numeric(cb_res[[1]]$data_info$outcome_info$sample_size), cb_res[[1]]$data_info$outcome_info$outcome_names)
cb_tab2[, cb_n := cb_n[paste0('DLPFC_Klein_gpQTL_unadjusted_', molecular_trait_id)]]; stopifnot(!anyNA(cb_tab2$cb_n))
cb_tab2[, cb_beta := colocboost:::get_beta(z_colocboost, cb_n)]
cb_tab2[, `:=`(cb_ci_lo = tanh(atanh(cb_beta) - qnorm(.975) / sqrt(cb_n - 3)), cb_ci_hi = tanh(atanh(cb_beta) + qnorm(.975) / sqrt(cb_n - 3)))]

TQ <- 'cis_association_corrected/unadjusted/unadj_glyco.sorted.bed.chr17_chr17.cis_qtl.pairs.tsv.gz'
COVF <- 'data_input/unadjust/covariate/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz'
tq_var <- sub(':([^:]+):([^:]+)$', '_\\1_\\2', lead_var5)   # chr:pos:ref:alt -> chr:pos_ref_alt (TensorQTL ID format)
tq <- fread(cmd = sprintf("zcat %s | awk -F'\t' 'NR==1||$4==\"%s\"'", TQ, tq_var))[molecular_trait_id %in% gp_ids5]
stopifnot(nrow(tq) == length(gp_ids5), all(tq$n == 316))
n_cov <- as.integer(system(sprintf('zcat %s | wc -l', COVF), intern = TRUE)) - 1L; cat('covariates:', n_cov, '\n')
tq[, dof := n - 2L - n_cov]                                  # TensorQTL residual degrees of freedom
stopifnot(isTRUE(all.equal(2 * pt(-abs(tq$bhat / tq$sebhat), tq$dof), tq$pvalue, tolerance = 1e-2)))   # recompute P from bhat/sebhat to validate dof and columns
tq[, `:=`(ci_lo = bhat - qt(.975, dof) * sebhat, ci_hi = bhat + qt(.975, dof) * sebhat)]   # t-based 95% CI, consistent with the TensorQTL P
cb_tab2 <- merge(cb_tab2, tq[, .(molecular_trait_id, beta = bhat, se = sebhat, ci_lo, ci_hi, pvalue, dof)], by = 'molecular_trait_id')

cb_tab2 <- merge(cb_tab2, anno_ref[gpID %in% gp_ids5, .(molecular_trait_id = gpID, peptide_ID, GlycanClass, GlycanComposition, GlycoSite)], by = 'molecular_trait_id')
cb_tab2[, GlycoSite := sub('.*_', '', GlycoSite)]                 # P12821_N111 -> N111
setorder(cb_tab2, molecular_trait_id)
saveRDS(cb_tab2, 'data/Figure_5a_colocboost_effect_ACE_glycopeptiforms.rds')
cb_tab2[, .(molecular_trait_id, GlycanComposition, coef_colocboost, cb_beta, z_colocboost, p_colocboost, beta, se, ci_lo, ci_hi, pvalue)]

fmt_ci <- function(b, lo, hi, d = 3) sprintf(paste0('%.', d, 'f [%.', d, 'f, %.', d, 'f]'), b, lo, hi)
lead_lab <- paste0('Lead variant ', lead_var5)
common <- cb_tab2[, .(`Glycan composition` = GlycanComposition, `Glycosite` = GlycoSite, `Glycan class` = GlycanClass)]
# set 1: TensorQTL (marginal beta per alternate allele in phenotype units, t-based CI
table2 <- cbind(common, cb_tab2[, setNames(list(fmt_ci(beta, ci_lo, ci_hi), signif(pvalue, 3)), paste(lead_lab, c('TensorQTL beta [95% CI]', 'TensorQTL P value')))])
fwrite(table2, 'data/Table2_ACE_glycopeptiforms.csv')
table2_cb <- cbind(common, cb_tab2[, setNames(list(fmt_ci(cb_beta, cb_ci_lo, cb_ci_hi), signif(p_colocboost, 3)), paste(lead_lab, c('colocBoost standardized beta [95% CI]', 'colocBoost P value')))])
fwrite(table2_cb, 'data/Table2_ACE_glycopeptiforms_colocboost.csv')
table2_cb